# Clase 3 — RAG (*retrieval augmented generation*)

*Arquitectura de Aplicaciones con IA Generativa — EscuelaIT · 90 min*

**Objetivo**: construir un RAG mínimo funcional entendiendo cada pieza desde cero, sin librerías que oculten el patrón.

**Flujo de la clase**

- Este notebook (`01-conceptos.ipynb`): tablas, diagramas, fórmulas — soporte visual para explicar.
- `02-demo-rag.ipynb`: implementación ejecutable.

Los bloques marcados **DEMO →** señalan cuándo cambiamos de archivo.

<style>
.jp-RenderedMarkdown { font-size: 1.15em; line-height: 1.5; }
.jp-RenderedMarkdown h1 { font-size: 2.1em; }
.jp-RenderedMarkdown h2 { font-size: 1.6em; margin-top: 0.6em; }
.jp-RenderedMarkdown h3 { font-size: 1.25em; }
.jp-RenderedMarkdown table { font-size: 0.95em; }
.jp-RenderedMarkdown th, .jp-RenderedMarkdown td { padding: 6px 10px; }
.jp-RenderedMarkdown pre { font-size: 0.9em; }
</style>

## Agenda

| # | Bloque | Min |
|---|---|---|
| 0 | Apertura y puente desde clase 2 | 3 |
| 1 | Cuatro límites del LLM puro | 4 |
| 2 | Prompting · RAG · Fine-tuning | 8 |
| 3 | Costes y casos de uso | 4 |
| 4 | Embeddings: qué son y cómo se entrenan | 8 |
| 5 | Similitud coseno | 4 |
| 6 | Anatomía de un RAG + glosario | 5 |
| 7 | **DEMO** — RAG desde cero con SQLite + MiniLM | 37 |
| 8 | Chunking y retrieval en producción | 4 |
| 9 | Modos de fallo y cuándo NO usar RAG | 3 |
| 10 | Cierre y preview clase 4 | 2 |

## §1 — Cuatro límites del LLM puro

| Límite | Qué significa | Ejemplo | Cómo lo ataca RAG |
|---|---|---|---|
| **Knowledge cutoff** | No sabe nada posterior a la fecha de entrenamiento | *"precio de BTC hoy"* | Inyecta datos frescos desde fuente externa |
| **Conocimiento privado** | Nunca vio tus datos internos | *"política interna de vacaciones"* | Indexa tus propios documentos |
| **Alucinaciones** | Genera por plausibilidad, no por certeza | Cita un paper que no existe | Ancla la respuesta a texto concreto recuperado |
| **Ventana de contexto** | Docs muy largos → caro, lento, *lost in the middle* | Manual de 400 páginas | Solo inyecta los chunks relevantes |

Los tres primeros son problemas de **conocimiento**. El cuarto es de **escala**. RAG ataca los cuatro por el mismo mecanismo.

## §2 — Tres formas de extender un LLM

| Dimensión | **Prompting** | **RAG** | **Fine-tuning** |
|---|---|---|---|
| Qué cambia | el texto del prompt | el contexto inyectado | los pesos del modelo |
| Frescura de datos | limitada al prompt | tiempo real (reindexas) | requiere reentrenar |
| Coste setup | horas | días | semanas + miles USD en GPU |
| Coste por consulta | bajo | bajo–medio | bajo (a veces menor que base) |
| Latencia | baja | +50–500 ms por retrieval | baja |
| Transparencia | total (el prompt se lee) | alta (ves los chunks) | baja (pesos opacos) |
| Actualizar conocimiento | editar prompt | reindexar | reentrenar |

### Heurística de decisión

```
¿funciona con prompting?  ──sí──► listo
         │
         no
         ▼
¿es problema de conocimiento? ──sí──► RAG
         │
         no (es de comportamiento/estilo)
         ▼
       Fine-tuning (último recurso)
```

**Antipattern frecuente**: *"voy a hacer fine-tuning para que sepa de mi empresa"* → casi siempre es RAG. Fine-tuning enseña **cómo** responder, no **qué** saber.

## §3 — Costes y casos de uso

### Coste comparado (orden de magnitud, referencial)

| Patrón | Setup | Por consulta | Modo de fallo típico |
|---|---|---|---|
| Prompting | horas | 0.001–0.01 USD | prompt crece sin límite si crece el conocimiento |
| RAG | días | ídem + embedding de query (~0.0001 USD) | retrieval malo → respuestas peores que sin RAG |
| Fine-tuning | semanas + miles USD | a veces menor (prompts más cortos) | cada cambio de requisitos → reentrenar |

### Dónde cae natural

| Patrón | Casos típicos |
|---|---|
| Prompting | clasificación, extracción, reformulación, razonamiento sobre input |
| RAG | Q&A sobre docs, asistentes internos, búsqueda semántica, asistentes legales/médicos con citas |
| Fine-tuning | estilo de escritura consistente, dominios cerrados con jerga propia, reducir tamaño de prompt |

## §4 — Qué es un embedding

> Un vector de números que representa el **significado** de un texto. Textos similares producen vectores cercanos en el espacio.

### Breve historia

| Año | Hito | Por qué importa |
|---|---|---|
| 2013 | **word2vec** | Primera aritmética semántica: `rey − hombre + mujer ≈ reina` |
| 2018+ | BERT, sentence-transformers | Embeddings de frases/párrafos, no solo palabras |
| 2024+ | BGE-m3, voyage-3, `text-embedding-3` | Multilingües, alta calidad, dimensiones variables |

### Cómo se entrena (contrastivo)

```
pares SIMILARES     ───►  los vectores deben ACERCARSE
(paráfrasis, traducciones, pregunta↔respuesta)

pares ALEATORIOS    ───►  los vectores deben ALEJARSE
```

Millones de pares → el espacio se organiza por significado.

### Densos vs dispersos

| Característica | Denso (embeddings modernos) | Disperso (TF-IDF, BM25) |
|---|---|---|
| Dimensiones | cientos (384–1536) | miles (tamaño del vocabulario) |
| Valores | reales, casi todos ≠ 0 | ceros en casi todas las dimensiones |
| Captura | significado, paráfrasis, sinónimos | coincidencia léxica exacta |
| Gana en | *"autos"* ≈ *"vehículos"* | códigos tipo *"ERP-2024-X"*, nombres propios |

En producción se combinan (**retrieval híbrido**). Hoy usamos solo denso.

### Dimensionalidad (más no siempre es mejor)

| Modelo | Dim | Notas |
|---|---|---|
| MiniLM multilingüe | 384 | el del demo, ~420 MB, rápido, local |
| OpenAI `text-embedding-3-small` | 1536 | API, barato |
| BGE-m3 | 1024 | el de antapaccay-demo, multilingüe |
| Cohere embed v3 | 1024 | API managed |

Para la mayoría de casos, 384–768 alcanza.

## §5 — Similitud coseno

Cómo medimos *cuán parecidos son* dos textos en el espacio de embeddings.

$$\text{sim}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \, \|\mathbf{b}\|} = \cos(\theta)$$

### Interpretación

| Valor | Significado |
|---|---|
| **1** | misma dirección — significado idéntico |
| **0** | ortogonales — nada en común |
| **−1** | direcciones opuestas (raro en la práctica) |

### Coseno vs distancia euclidiana

| | Coseno | Euclidiana |
|---|---|---|
| Qué mide | dirección (ángulo) | distancia absoluta |
| Efecto de la longitud del texto | ignora magnitud | se ve afectada |
| Dos textos iguales de distinta longitud | similitud alta | puede dar lejos |

### Truco estándar: normalizar a norma 1

Si ambos vectores tienen norma 1, el coseno se reduce al producto punto:

$$\text{sim}(\mathbf{a}, \mathbf{b}) = \mathbf{a} \cdot \mathbf{b} \quad \text{cuando } \|\mathbf{a}\| = \|\mathbf{b}\| = 1$$

Eso es lo que hacemos en el demo: `normalize_embeddings=True` al embeber, y un solo `np.dot` para la similitud.

## §6 — Anatomía de un sistema RAG

Dos fases, temporalmente separadas.

### Ingesta (offline, una vez o periódica)

```
documentos  →  chunking  →  embedder  →  vector_store (persistente)
```

### Consulta (online, por cada pregunta)

```
pregunta
  ├──► embedder → query_vec ──┐
  │                           ▼
  │                      retrieval (top-K similar)
  │                           │
  └─────────────────────►  prompt = instrucciones + chunks + pregunta
                                          │
                                          ▼
                                         LLM
                                          │
                                          ▼
                                    respuesta + citas
```

### Glosario rápido

| Concepto | Qué es | Valor típico |
|---|---|---|
| **Chunk** | fragmento de documento | 200–800 tokens |
| **Vector store** | BD con búsqueda por similitud | SQLite + BLOB → Pinecone |
| **Retrieval** | top-K más similares a la query | K = 3–10 |
| **Augmented generation** | inyectar chunks en prompt + LLM | — |
| **Citaciones** | referencias a chunks usados en la respuesta | para auditoría |

---

## DEMO → `02-demo-rag.ipynb`

Implementamos el ciclo completo:

- **SQLite** como vector store (embeddings como BLOB)
- **MiniLM** como embedder (local, 384 dim)
- **NumPy** para la similitud coseno
- **DeepSeek** como LLM

Al volver: chunking en producción, modos de fallo, cuándo NO usar RAG, y cierre.

---

## §7 — Chunking en producción

### Estrategias

| Estrategia | Cómo funciona | Ventaja | Desventaja |
|---|---|---|---|
| **Fixed-size** | N caracteres/tokens con overlap | simple, predecible | corta a mitad de frase |
| **Por párrafos** | respeta `\n\n` | respeta estructura natural | tamaños desiguales |
| **Recursive** | separadores en orden: `\n\n` → `\n` → `. ` → ` ` | balance estructura + tamaño | más complejo |
| **Semantic** | parte donde cambia el tema (coseno entre frases) | chunks coherentes | caro (embeddings extra) |
| **Por estructura** | respeta headings MD/HTML | ideal para docs técnicos | requiere parser |

### Tradeoffs de tamaño

| Tamaño | Pros | Contras |
|---|---|---|
| Pequeño (100–200 tok) | retrieval fino, menos ruido | pierde contexto, referencias colgadas |
| Medio (400–800 tok) | *sweet spot* para la mayoría | — |
| Grande (1500+ tok) | contexto completo por chunk | embedding difuso, pocos chunks caben en prompt |

## §8 — Retrieval en producción

### Elección de K

| K | Cuándo | Riesgo |
|---|---|---|
| 1–3 | respuestas precisas, corpus bien estructurado | deja fuera info relevante |
| 5–10 | Q&A general | ruido, *lost in the middle* |
| 20+ | solo como entrada a reranking | caro si va directo al LLM |

### Refinamientos típicos

| Técnica | Qué hace | Cuándo ayuda |
|---|---|---|
| **Retrieval híbrido** | denso + disperso (BM25) fusionado con RRF | significado + coincidencia exacta |
| **Reranking** | cross-encoder re-ordena top-20 → top-3 | cuando el orden del top importa |
| **Query rewriting** | LLM reformula la query antes de embeber | expandir siglas, desambiguar |
| **HyDE** | generar respuesta hipotética y embeber **eso** | query "suena" más como los docs |
| **Metadata filters** | filtrar por autor/fecha/sección antes del coseno | precisión + menos coste |

## §9 — Modos de fallo comunes

| Fallo | Síntoma visible | Causa raíz |
|---|---|---|
| Chunks cortados a mitad | respuesta inconexa, referencias rotas | chunking por caracteres sin respetar frases |
| Top-K irrelevante | LLM inventa con tono confiado | modelo de embeddings mal calibrado al dominio |
| Alucinación **con** RAG | usa el contexto y además añade mentiras | system prompt sin "di *no sé*" explícito |
| Similitud baja para todo | nada se recupera con confianza | queries cortas vs docs largos → HyDE, query rewriting |
| Idioma inconsistente | a veces responde en inglés | modelo de embedding/LLM no tuned al idioma del corpus |
| Retrieval lento (p95 > 1 s) | latencia alta en producción | O(n) en Python; falta índice HNSW/IVF |

## §10 — Cuándo NO usar RAG

| Situación | Qué hacer en su lugar |
|---|---|
| Tu corpus cabe en el prompt (<100K tokens) | inyéctalo directo — más simple y fiel |
| El LLM ya sabe la respuesta (hechos públicos) | solo prompting |
| Tarea de razonamiento puro (clasificar, traducir) | no necesita *buscar*, necesita *procesar* |
| No puedes medir la calidad del retrieval | mide primero — RAG malo es peor que sin RAG |
| Latencia crítica (< 200 ms) | RAG añade 50–500 ms — evaluar si compensa |

### Regla mental

> **RAG** resuelve problemas de *qué saber*.
>
> **Prompting** resuelve problemas de *cómo responder*.
>
> **Fine-tuning** resuelve problemas de *comportamiento del modelo*.

## Cierre

### Tres ideas para llevarse

1. **RAG = búsqueda semántica + inyección de contexto.** Todo lo demás es optimización sobre ese núcleo.
2. **Prompting, RAG y fine-tuning responden preguntas distintas.** *Cómo* responder, *qué* saber, *comportamiento* del modelo.
3. **Lo que hicimos hoy vive en los sistemas reales.** Antapaccay-demo = este patrón + retrieval híbrido + reranking + citaciones.

### Preview — Clase 4

El RAG de hoy → **aplicación real**.

| Capa | Stack |
|---|---|
| Backend | FastAPI |
| Streaming | Server-Sent Events |
| Persistencia | SQLite / Postgres |
| Frontend | React o similar |
| Transversal | rate limiting, auth, observabilidad |